## Problem statement
Implement the forward and backward pass of a many-to-one RNN (sequence classification) with a toy input of two time steps $(x_1, x_2)$, and compare with PyTorch.

$$h_t=\tanh(W_x x_t+W_h h_{t-1}+b_h),\qquad o_T=W_o h_T+b_o,\qquad \hat y=\text{softmax}(o_T)$$

Loss: cross-entropy $L=-\sum_k y_k\log \hat y_k$ (one-hot target; this is exactly what `torch.nn.CrossEntropyLoss` computes, so the two implementations are directly comparable).

Dimensions: $n_x=1$, $n_h=1$, $n_y=2$, $h_0=0$.

In [1]:
import numpy as np
np.set_printoptions(precision=8, suppress=True)

## S1: Initialization

In [2]:
np.random.seed(0)
nx, nh, ny = 1, 1, 2

Wx = np.random.randn(nh, nx) * 0.5     # (nh, nx)
Wh = np.random.randn(nh, nh) * 0.5     # (nh, nh)
bh = np.random.randn(nh, 1) * 0.1      # (nh, 1)
Wo = np.random.randn(ny, nh) * 0.5     # (ny, nh)
bo = np.random.randn(ny, 1) * 0.1      # (ny, 1)
h0 = np.zeros((nh, 1))

# toy sequence (x1, x2) and one-hot target (class 1)
x1 = np.array([[0.5]])
x2 = np.array([[-0.8]])
xs = [x1, x2]
y  = np.array([[0.0], [1.0]])
eta = 0.1                              # SGD learning rate

P = [Wx, Wh, bh, Wo, bo]
P_init = [p.copy() for p in P]         # keep initial values to copy into PyTorch later
names = ["Wx", "Wh", "bh", "Wo", "bo"]
for n, p in zip(names, P): print(n, "=", p.ravel())

Wx = [0.88202617]
Wh = [0.2000786]
bh = [0.0978738]
Wo = [1.1204466 0.933779 ]
bo = [-0.09772779  0.09500884]


## S2: Forward pass
$h_1=\tanh(W_x x_1+W_h h_0+b_h)$, $h_2=\tanh(W_x x_2+W_h h_1+b_h)$, $o_2=W_o h_2+b_o$, $\hat y=\text{softmax}(o_2)$.
Only the last time step contributes to the loss (many-to-one).

In [3]:
def softmax(o):
    e = np.exp(o - o.max())
    return e / e.sum()

def forward(P, xs, y):
    Wx, Wh, bh, Wo, bo = P
    hs = [h0]
    for x in xs:
        hs.append(np.tanh(Wx @ x + Wh @ hs[-1] + bh))
    o = Wo @ hs[-1] + bo
    yhat = softmax(o)
    L = -np.sum(y * np.log(yhat))
    return L, yhat, hs, o

L0, yhat, hs, o2 = forward(P, xs, y)
print("h1 =", hs[1].ravel(), " h2 =", hs[2].ravel())
print("o2 =", o2.ravel(), " y_hat =", yhat.ravel())
print("Loss before update L0 =", L0)

h1 = [0.49214492]  h2 = [-0.46938361]
o2 = [-0.62364706 -0.34329171]  y_hat = [0.43036666 0.56963334]
Loss before update L0 = 0.5627623900895352


## S3: Backward pass (BPTT) — derivation
Let $\hat y=\text{softmax}(o_2)$. For softmax + cross-entropy, $\dfrac{\partial L}{\partial o_2}=\hat y-y$.

* $\dfrac{\partial L}{\partial W_o}=(\hat y-y)h_2^T$, $\dfrac{\partial L}{\partial b_o}=\hat y-y$, $\dfrac{\partial L}{\partial h_2}=W_o^T(\hat y-y)$.
* Since $h_t=\tanh(a_t)$, $\dfrac{\partial h_t}{\partial a_t}=1-h_t^2$. Define $\delta_t=\dfrac{\partial L}{\partial h_t}\odot(1-h_t^2)$ (gradient w.r.t. pre-activation $a_t$).
* Gradient flows back through the recurrence: $\dfrac{\partial L}{\partial h_{1}}=W_h^T\delta_2$, so $\delta_1=\big(W_h^T\delta_2\big)\odot(1-h_1^2)$.
  (The lab sheet writes this as $W_h^T\frac{\partial L}{\partial h_2}\odot(1-h_2^2)$; the correct order is to multiply by $(1-h_2^2)$ *first* and then apply $W_h^T$, which is what is done here and what PyTorch autograd does.)
* Parameters are shared over time, so gradients accumulate: $\dfrac{\partial L}{\partial W_x}=\sum_t\delta_t x_t^T$, $\dfrac{\partial L}{\partial W_h}=\sum_t\delta_t h_{t-1}^T$, $\dfrac{\partial L}{\partial b_h}=\sum_t\delta_t$.

In [4]:
def backward(P, xs, y, hs, yhat):
    Wx, Wh, bh, Wo, bo = P
    do  = yhat - y                       # dL/do2
    dWo = do @ hs[-1].T
    dbo = do.copy()
    dh  = Wo.T @ do                      # dL/dh2
    dWx, dWh, dbh = np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(bh)
    for t in range(len(xs), 0, -1):      # t = T ... 1
        delta = dh * (1 - hs[t] ** 2)    # dL/da_t
        dWx += delta @ xs[t - 1].T
        dWh += delta @ hs[t - 1].T
        dbh += delta
        dh = Wh.T @ delta                # dL/dh_{t-1}
    return [dWx, dWh, dbh, dWo, dbo]

grads = backward(P, xs, y, hs, yhat)
for n, g in zip(names, grads): print("d" + n, "=", g.ravel())

dWx = [-0.04536036]
dWh = [0.03082595]
dbh = [0.07213266]
dWo = [-0.20200706  0.20200706]
dbo = [ 0.43036666 -0.43036666]


### Verification of the derived gradients with a numerical (finite-difference) check

In [5]:
def num_grad(P, k, eps=1e-6):
    g = np.zeros_like(P[k])
    for idx in np.ndindex(*P[k].shape):
        Pp = [p.copy() for p in P]; Pm = [p.copy() for p in P]
        Pp[k][idx] += eps; Pm[k][idx] -= eps
        g[idx] = (forward(Pp, xs, y)[0] - forward(Pm, xs, y)[0]) / (2 * eps)
    return g

for k, n in enumerate(names):
    ng = num_grad(P, k)
    print(f"{n}: analytic={grads[k].ravel()}  numeric={ng.ravel()}  match={np.allclose(grads[k], ng, atol=1e-6)}")

Wx: analytic=[-0.04536036]  numeric=[-0.04536036]  match=True
Wh: analytic=[0.03082595]  numeric=[0.03082595]  match=True
bh: analytic=[0.07213266]  numeric=[0.07213266]  match=True
Wo: analytic=[-0.20200706  0.20200706]  numeric=[-0.20200706  0.20200706]  match=True
bo: analytic=[ 0.43036666 -0.43036666]  numeric=[ 0.43036666 -0.43036666]  match=True


## S4: Parameter update (one SGD step, $\theta\leftarrow\theta-\eta\,\partial L/\partial\theta$)

In [6]:
P_new = [p - eta * g for p, g in zip(P, grads)]
L1, yhat1, _, _ = forward(P_new, xs, y)
print("Loss before update:", L0)
print("Loss after  update:", L1)
print("Loss decreased:", L1 < L0)
for n, p in zip(names, P_new): print(n, "(updated) =", p.ravel())

Loss before update: 0.5627623900895352
Loss after  update: 0.5179707032625847
Loss decreased: True
Wx (updated) = [0.88656221]
Wh (updated) = [0.19699601]
bh (updated) = [0.09066053]
Wo (updated) = [1.14064731 0.91357829]
bo (updated) = [-0.14076445  0.13804551]


## S5: PyTorch implementation
`torch.nn.RNN` has two bias vectors (`bias_ih_l0`, `bias_hh_l0`); the effective bias is their sum. To match our single $b_h$, we copy $b_h$ into `bias_ih_l0`, set `bias_hh_l0 = 0` and freeze it. Double precision is used so the match is to ~1e-15.

In [7]:
import torch, torch.nn as nn
dt = torch.float64
T = lambda a: torch.tensor(a, dtype=dt)

rnn = nn.RNN(input_size=1, hidden_size=1, num_layers=1, nonlinearity="tanh", dtype=dt)
fc  = nn.Linear(nh, ny, dtype=dt)          # output layer (Wo, bo)

with torch.no_grad():
    rnn.weight_ih_l0.copy_(T(P_init[0]))             # Wx
    rnn.weight_hh_l0.copy_(T(P_init[1]))             # Wh
    rnn.bias_ih_l0.copy_(T(P_init[2].ravel()))       # bh
    rnn.bias_hh_l0.zero_()
    fc.weight.copy_(T(P_init[3]))                    # Wo
    fc.bias.copy_(T(P_init[4].ravel()))              # bo
rnn.bias_hh_l0.requires_grad_(False)

X = T(np.array([x1.item(), x2.item()])).view(2, 1, 1)   # (T, batch, features)
target = torch.tensor([int(np.argmax(y))])
criterion = nn.CrossEntropyLoss()
opt = torch.optim.SGD([p for p in list(rnn.parameters()) + list(fc.parameters()) if p.requires_grad], lr=eta)

out, hn = rnn(X)
logits = fc(out[-1])                       # many-to-one: last time step only
loss = criterion(logits, target)
opt.zero_grad(); loss.backward()
torch_L0 = loss.item()
torch_grads = [rnn.weight_ih_l0.grad.numpy().copy(), rnn.weight_hh_l0.grad.numpy().copy(),
               rnn.bias_ih_l0.grad.numpy().copy().reshape(-1, 1),
               fc.weight.grad.numpy().copy(), fc.bias.grad.numpy().copy().reshape(-1, 1)]
opt.step()
with torch.no_grad():
    out2, _ = rnn(X); torch_L1 = criterion(fc(out2[-1]), target).item()
torch_P = [rnn.weight_ih_l0, rnn.weight_hh_l0, rnn.bias_ih_l0, fc.weight, fc.bias]
torch_P = [p.detach().numpy().copy().reshape(q.shape) for p, q in zip(torch_P, P)]
print("PyTorch loss before:", torch_L0, " after:", torch_L1)

PyTorch loss before: 0.5627623900895351  after: 0.5179707032625847


## S6: Comparison of the two implementations

In [8]:
print(f"{'':22s}{'NumPy':>22s}{'PyTorch':>22s}")
print(f"{'Loss before update':22s}{L0:22.12f}{torch_L0:22.12f}")
print(f"{'Loss after update':22s}{L1:22.12f}{torch_L1:22.12f}")
print("\n--- Gradients ---")
for n, g, tg in zip(names, grads, torch_grads):
    print(f"d{n:<4s} NumPy={g.ravel()}  PyTorch={tg.ravel()}  match={np.allclose(g, tg, atol=1e-10)}")
print("\n--- Updated parameters ---")
for n, p, tp in zip(names, P_new, torch_P):
    print(f"{n:<4s} NumPy={p.ravel()}  PyTorch={tp.ravel()}  match={np.allclose(p, tp, atol=1e-10)}")

                                       NumPy               PyTorch
Loss before update            0.562762390090        0.562762390090
Loss after update             0.517970703263        0.517970703263

--- Gradients ---
dWx   NumPy=[-0.04536036]  PyTorch=[-0.04536036]  match=True
dWh   NumPy=[0.03082595]  PyTorch=[0.03082595]  match=True
dbh   NumPy=[0.07213266]  PyTorch=[0.07213266]  match=True
dWo   NumPy=[-0.20200706  0.20200706]  PyTorch=[-0.20200706  0.20200706]  match=True
dbo   NumPy=[ 0.43036666 -0.43036666]  PyTorch=[ 0.43036666 -0.43036666]  match=True

--- Updated parameters ---
Wx   NumPy=[0.88656221]  PyTorch=[0.88656221]  match=True
Wh   NumPy=[0.19699601]  PyTorch=[0.19699601]  match=True
bh   NumPy=[0.09066053]  PyTorch=[0.09066053]  match=True
Wo   NumPy=[1.14064731 0.91357829]  PyTorch=[1.14064731 0.91357829]  match=True
bo   NumPy=[-0.14076445  0.13804551]  PyTorch=[-0.14076445  0.13804551]  match=True


### Conclusion
The forward loss, all five parameter gradients (also confirmed by finite differences) and the parameters after one SGD step agree between the scratch NumPy implementation and PyTorch up to floating-point round-off. The loss decreases after the update, confirming that the BPTT gradients are correct.